In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window


In [0]:
# Paths
BRONZE_FLIGHTS_PATH = "/Volumes/.../bronze_flights"
BRONZE_AIRPORTS_PATH = "/Volumes/.../bronze_airports"

SILVER_OUTPUT_PATH = "/Volumes/.../silver_flights"

In [0]:
# Read Bronze tables
df = spark.read.format("delta").table('bronze_flights')
airlines_df = spark.read.format("delta").table('bronze_airports')
airports_df = spark.read.format("delta").table('bronze_airports')

### Parse Time Columns

In [0]:
def parse_hhmm(col_name):
    return lpad(col(col_name).cast("string"), 4, "0")

df = df.withColumn("sched_dep_str", parse_hhmm("SCHEDULED_DEPARTURE")) \
       .withColumn("dep_time_str", parse_hhmm("DEPARTURE_TIME")) \
       .withColumn("sched_arr_str", parse_hhmm("SCHEDULED_ARRIVAL")) \
       .withColumn("arr_time_str", parse_hhmm("ARRIVAL_TIME"))

df = df.withColumn(
    "scheduled_departure_ts",
    to_timestamp(concat_ws(" ",
        concat_ws("-", "YEAR", "MONTH", "DAY"),
        concat_ws(":", substring("sched_dep_str",1,2), substring("sched_dep_str",3,2))
    ))
).withColumn(
    "actual_departure_ts",
    to_timestamp(concat_ws(" ",
        concat_ws("-", "YEAR", "MONTH", "DAY"),
        concat_ws(":", substring("dep_time_str",1,2), substring("dep_time_str",3,2))
    ))
).withColumn(
    "scheduled_arrival_ts",
    to_timestamp(concat_ws(" ",
        concat_ws("-", "YEAR", "MONTH", "DAY"),
        concat_ws(":", substring("sched_arr_str",1,2), substring("sched_arr_str",3,2))
    ))
).withColumn(
    "actual_arrival_ts",
    to_timestamp(concat_ws(" ",
        concat_ws("-", "YEAR", "MONTH", "DAY"),
        concat_ws(":", substring("arr_time_str",1,2), substring("arr_time_str",3,2))
    ))
)


### Derived Columns

In [0]:
df = df.withColumn("flight_date", to_date(concat_ws("-", "YEAR","MONTH","DAY"))) \
       .withColumn("departure_hour", hour("scheduled_departure_ts")) \
       .withColumn("arrival_hour", hour("scheduled_arrival_ts")) \
       .withColumn("is_weekend", col("DAY_OF_WEEK").isin([6,7])) \
       .withColumn("flight_quarter", concat(lit("Q"), quarter("flight_date")))

### Airport Standardization + Enrichment

In [0]:
airports_clean = airports_df.select(
    col("IATA_CODE").alias("iata"),
    col("CITY"),
    col("STATE"),
    col("LATITUDE"),
    col("LONGITUDE")
)

# Origin join
df = df.join(
    airports_clean.withColumnRenamed("iata","origin_iata"),
    df.ORIGIN_AIRPORT == col("origin_iata"),
    "left"
).withColumnRenamed("CITY","origin_city") \
 .withColumnRenamed("STATE","origin_state") \
 .withColumnRenamed("LATITUDE","origin_lat") \
 .withColumnRenamed("LONGITUDE","origin_lon")

# Destination join
df = df.join(
    airports_clean.withColumnRenamed("iata","dest_iata"),
    df.DESTINATION_AIRPORT == col("dest_iata"),
    "left"
).withColumnRenamed("CITY","dest_city") \
 .withColumnRenamed("STATE","dest_state") \
 .withColumnRenamed("LATITUDE","dest_lat") \
 .withColumnRenamed("LONGITUDE","dest_lon")

# Unknown airport flag
df = df.withColumn(
    "is_unknown_airport",
    when(col("origin_city").isNull() | col("dest_city").isNull(), True).otherwise(False)
)


### Airline Enrichment

In [0]:
df = df.join(
    airlines_df,
    df.AIRLINE == airlines_df.IATA_CODE,
    "left"
).withColumnRenamed("AIRLINE_y", "airline_full_name")

### Delay Category

In [0]:
df = df.withColumn(
    "delay_category",
    when(col("CANCELLED") == 1, None)
    .when(col("ARRIVAL_DELAY") <= 0, "on_time")
    .when(col("ARRIVAL_DELAY") <= 15, "slight")
    .when(col("ARRIVAL_DELAY") <= 60, "moderate")
    .otherwise("severe")
)

### Cancellation Mapping

In [0]:
df = df.withColumn(
    "cancellation_reason_mapped",
    when(col("CANCELLATION_REASON") == "A", "Airline/Carrier")
    .when(col("CANCELLATION_REASON") == "B", "Weather")
    .when(col("CANCELLATION_REASON") == "C", "National Air System")
    .when(col("CANCELLATION_REASON") == "D", "Security")
)

### Fill Delay Nulls

In [0]:
delay_cols = [
    "WEATHER_DELAY", "AIRLINE_DELAY", "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY"
]

for c in delay_cols:
    df = df.withColumn(c, coalesce(col(c), lit(0.0)))

### Metrics + Flags

In [0]:
df = df.withColumn(
    "total_delay_minutes",
    col("WEATHER_DELAY") + col("AIRLINE_DELAY") +
    col("AIR_SYSTEM_DELAY") + col("SECURITY_DELAY") +
    col("LATE_AIRCRAFT_DELAY")
)

df = df.withColumn(
    "is_time_anomaly",
    (col("ELAPSED_TIME") <= 0) | (col("AIR_TIME") <= 0)
)

df = df.withColumn(
    "TAIL_NUMBER",
    coalesce(col("TAIL_NUMBER"), lit("UNKNOWN"))
)

### Duplicate Detection

In [0]:
window_spec = Window.partitionBy(
    "FLIGHT_NUMBER","ORIGIN_AIRPORT","DESTINATION_AIRPORT","flight_date"
)

df = df.withColumn("dup_count", count("*").over(window_spec)) \
       .withColumn("is_duplicate", col("dup_count") > 1)

### Business Columns

In [0]:
df = df.withColumn(
    "taxi_total_minutes",
    coalesce(col("TAXI_OUT"), lit(0)) + coalesce(col("TAXI_IN"), lit(0))
)

df = df.withColumn(
    "route_id",
    concat_ws("-", col("ORIGIN_AIRPORT"), col("DESTINATION_AIRPORT"))
)

### Write Final Silver Table

In [0]:
df.write \
  .format("delta") \
  .mode("overwrite") \
  .partitionBy("flight_date") \
  .save(silver_output_path)